# Hugging Face Dataset Basics for NLP

This notebook focuses on dataset loading, inspection, cleaning, and baseline modeling with `ag_news`.

In [ ]:
%pip install -q datasets scikit-learn pandas

In [ ]:
from datasets import load_dataset
import pandas as pd

ds = load_dataset("ag_news")
ds

In [ ]:
labels = ds["train"].features["label"].names
labels

In [ ]:
df_train = ds["train"].to_pandas()
df_test = ds["test"].to_pandas()

df_train.head()

In [ ]:
# Map numeric labels to names
df_train["label_name"] = df_train["label"].map(lambda x: labels[x])
df_train["label_name"].value_counts()

In [ ]:
# Basic text stats
df_train["n_chars"] = df_train["text"].str.len()
df_train["n_words"] = df_train["text"].str.split().map(len)
df_train[["n_chars", "n_words"]].describe()

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report

sample_train = ds["train"].shuffle(seed=123).select(range(8000))
sample_test = ds["test"].shuffle(seed=123).select(range(2000))

x_train = sample_train["text"]
y_train = sample_train["label"]
x_test = sample_test["text"]
y_test = sample_test["label"]

vectorizer = TfidfVectorizer(max_features=30000, ngram_range=(1,2))
x_train_vec = vectorizer.fit_transform(x_train)
x_test_vec = vectorizer.transform(x_test)

clf = LogisticRegression(max_iter=400)
clf.fit(x_train_vec, y_train)
pred = clf.predict(x_test_vec)

print(classification_report(y_test, pred, target_names=labels))

## Next Practice Ideas

1. Swap `ag_news` with `imdb`, `yelp_polarity`, or `tweet_eval`.
2. Compare `CountVectorizer` vs `TfidfVectorizer`.
3. Feed the same train/test split into a Transformer classifier.